# RQ3 Colab Runner

This notebook clones `kiriennn/tgr`, prepares Amazon Beauty data, builds Semantic IDs, and runs Research Question 3: whether explicit time-gap modeling improves TIGER generative retrieval.

All generated data, logs, metrics, result JSONs, and checkpoints are stored under Google Drive. If Colab disconnects or a run stops before finishing, rerun the notebook cells with `FORCE_RERUN = False`; RQ-VAE and TIGER training resume from their latest saved checkpoints.

In [ ]:
REPO_URL = "https://github.com/kiriennn/tgr.git"
BRANCH = None
PROJECT_DIR = "/content/tgr"

USE_DRIVE_CACHE = True
DRIVE_ROOT = "/content/drive/MyDrive/tgr-rq3-colab"

CATEGORY = "Beauty"
CODEBOOK_SIZE = 256
RQVAE_EPOCHS = 500
RQVAE_CHECKPOINT_EVERY = 20
TIGER_EPOCHS = 100
TIGER_CHECKPOINT_EVERY = 1
LR = "3e-4"
MAX_ITEMS = 20
NUM_BEAMS = 100
BATCH_SIZE = 256
NUM_WORKERS = 2
PIN_MEMORY = True
DEVICE = "cuda"
SEED = 42
FORCE_RERUN = False

DATA_DIR = f"data/processed/{CATEGORY}"
TEMP_DATA_DIR = f"data/processed/{CATEGORY}_temporal"
CODES = f"{DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}.npy"
TEMP_CODES = f"{TEMP_DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}_fit-temporal.npy"
RUN_DIR = "runs/rq3"
LOG_DIR = f"{RUN_DIR}/logs"
METRICS_DIR = f"{RUN_DIR}/metrics"
CHECKPOINT_DIR = f"{RUN_DIR}/checkpoints"

In [ ]:
!nvidia-smi || true
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess

def sh(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

if not os.path.exists(PROJECT_DIR):
    branch_arg = "" if BRANCH is None else f"--branch {BRANCH} "
    sh(f"git clone {branch_arg}{REPO_URL} {PROJECT_DIR}")
else:
    sh("git fetch origin", cwd=PROJECT_DIR)
    if BRANCH is not None:
        sh(f"git checkout {BRANCH}", cwd=PROJECT_DIR)
    sh("git pull --ff-only", cwd=PROJECT_DIR)

%cd {PROJECT_DIR}
!git status --short --branch
!git log --oneline -3

In [ ]:
import pathlib
import shutil

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    root = pathlib.Path(DRIVE_ROOT)
    for sub in ["data/raw", "data/processed", "runs"]:
        (root / sub).mkdir(parents=True, exist_ok=True)
    pathlib.Path("data").mkdir(exist_ok=True)
    for local, target in [
        ("data/raw", root / "data/raw"),
        ("data/processed", root / "data/processed"),
        ("runs", root / "runs"),
    ]:
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.exists(local):
            shutil.rmtree(local)
        os.symlink(target, local)

for path in [RUN_DIR, LOG_DIR, METRICS_DIR, CHECKPOINT_DIR]:
    os.makedirs(path, exist_ok=True)

!ls -lah data runs

In [ ]:
!python -m pip install -q -r requirements.txt
!python - <<'PY'
import torch, transformers, sklearn
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('sklearn', sklearn.__version__)
PY

In [ ]:
import glob
import json
import time

def run_logged(args, out_path=None, log_path=None, force=False):
    if out_path and os.path.exists(out_path) and not force:
        print(f"[skip] {out_path} exists")
        return
    if out_path:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
    if log_path:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
    print("$ " + " ".join(map(str, args)))
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    started = time.time()
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    with open(log_path, "a", encoding="utf-8") if log_path else open(os.devnull, "w") as log:
        log.write(f"\n===== command started {time.ctime(started)} =====\n")
        log.write("$ " + " ".join(map(str, args)) + "\n")
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
    rc = proc.wait()
    print(f"[exit {rc}] elapsed={(time.time() - started) / 60:.1f} min")
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}")

def run_json(name):
    return f"{RUN_DIR}/{CATEGORY}_{name}_lr3e4_e100_seed{SEED}.json"

def run_log(name):
    return f"{LOG_DIR}/{CATEGORY}_{name}_lr3e4_e100_seed{SEED}.log"

def metrics_log(name):
    return f"{METRICS_DIR}/{CATEGORY}_{name}_lr3e4_e100_seed{SEED}.jsonl"

def checkpoint_dir(name):
    return f"{CHECKPOINT_DIR}/{name}_seed{SEED}"

## Status Before Running

Use this cell after reconnecting to see which outputs already exist and which checkpoints are available.

In [ ]:
def checkpoint_epoch(path):
    if not os.path.exists(path):
        return None
    try:
        state = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        state = torch.load(path, map_location="cpu")
    return state.get("epoch")

status = []
done_paths = {
    "semantic_ids_leave_one_out": f"{DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}.meta.json",
    "semantic_ids_temporal": f"{TEMP_DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}_fit-temporal.meta.json",
}
for name in ["semantic_ids_leave_one_out", "semantic_ids_temporal", "tiger_leave_one_out", "tiger_temporal", "tiger_time_gaps_temporal", "tiger_time_embed_temporal"]:
    path = checkpoint_dir(name)
    last = os.path.join(path, "rqvae_last.pt") if name.startswith("semantic_ids") else os.path.join(path, "last.pt")
    done = os.path.exists(done_paths.get(name, run_json(name)))
    status.append({"name": name, "checkpoint": last, "epoch": checkpoint_epoch(last), "done": done})

try:
    import pandas as pd
    display(pd.DataFrame(status))
except Exception:
    for row in status:
        print(row)

## Data And Semantic IDs

In [ ]:
run_logged(["python", "src/data/download.py", "--category", CATEGORY, "--out", "data/raw", "--review-set", "5core"], log_path=f"{LOG_DIR}/download_5core.log", force=True)
run_logged(["python", "src/data/download.py", "--category", CATEGORY, "--out", "data/raw", "--review-set", "all"], log_path=f"{LOG_DIR}/download_all.log", force=True)

run_logged(["python", "src/data/preprocess.py", "--category", CATEGORY, "--raw", "data/raw", "--out", DATA_DIR, "--kcore", "5"], out_path=f"{DATA_DIR}/meta.json", log_path=f"{LOG_DIR}/preprocess_leave_one_out.log", force=FORCE_RERUN)
run_logged(["python", "src/data/preprocess.py", "--category", CATEGORY, "--raw", "data/raw", "--out", TEMP_DATA_DIR, "--kcore", "5", "--kcore-scope", "train_temporal", "--review-set", "all"], out_path=f"{TEMP_DATA_DIR}/meta.json", log_path=f"{LOG_DIR}/preprocess_temporal.log", force=FORCE_RERUN)

print(json.dumps(json.load(open(f"{DATA_DIR}/meta.json")), indent=2))
print(json.dumps(json.load(open(f"{TEMP_DATA_DIR}/meta.json")), indent=2))

In [ ]:
run_logged([
    "python", "-u", "src/train_semantic_ids.py",
    "--data", DATA_DIR,
    "--id-method", "rqvae",
    "--num-levels", "3",
    "--codebook-size", str(CODEBOOK_SIZE),
    "--rqvae-epochs", str(RQVAE_EPOCHS),
    "--checkpoint-dir", checkpoint_dir("semantic_ids_leave_one_out"),
    "--checkpoint-every", str(RQVAE_CHECKPOINT_EVERY),
    "--resume",
    "--device", DEVICE,
    "--seed", str(SEED),
], out_path=f"{DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}.meta.json", log_path=run_log("semantic_ids_leave_one_out"), force=FORCE_RERUN)

run_logged([
    "python", "-u", "src/train_semantic_ids.py",
    "--data", TEMP_DATA_DIR,
    "--id-method", "rqvae",
    "--num-levels", "3",
    "--codebook-size", str(CODEBOOK_SIZE),
    "--rqvae-epochs", str(RQVAE_EPOCHS),
    "--fit-split", "temporal",
    "--checkpoint-dir", checkpoint_dir("semantic_ids_temporal"),
    "--checkpoint-every", str(RQVAE_CHECKPOINT_EVERY),
    "--resume",
    "--device", DEVICE,
    "--seed", str(SEED),
], out_path=f"{TEMP_DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}_fit-temporal.meta.json", log_path=run_log("semantic_ids_temporal"), force=FORCE_RERUN)

!ls -lh data/processed/Beauty/codes_rqvae_L3_K256.npy data/processed/Beauty_temporal/codes_rqvae_L3_K256_fit-temporal.npy

## RQ3 Training Runs

In [ ]:
def tiger_args(data_dir, codes, split, name, extra=None):
    args = [
        "python", "-u", "src/train_tiger.py",
        "--data", data_dir,
        "--codes", codes,
        "--codebook-size", str(CODEBOOK_SIZE),
        "--max-items", str(MAX_ITEMS),
        "--split", split,
        "--epochs", str(TIGER_EPOCHS),
        "--lr", LR,
        "--batch-size", str(BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS),
        "--num-beams", str(NUM_BEAMS),
        "--checkpoint-dir", checkpoint_dir(name),
        "--checkpoint-every", str(TIGER_CHECKPOINT_EVERY),
        "--resume",
        "--metrics-log", metrics_log(name),
        "--device", DEVICE,
        "--seed", str(SEED),
        "--out", run_json(name),
    ]
    if PIN_MEMORY:
        args.append("--pin-memory")
    if extra:
        args.extend(extra)
    return args

def run_tiger(name, data_dir, codes, split, extra=None):
    run_logged(tiger_args(data_dir, codes, split, name, extra), out_path=run_json(name), log_path=run_log(name), force=FORCE_RERUN)

In [ ]:
run_tiger("tiger_leave_one_out", DATA_DIR, CODES, "leave_one_out")

In [ ]:
run_tiger("tiger_temporal", TEMP_DATA_DIR, TEMP_CODES, "temporal")

In [ ]:
run_tiger("tiger_time_gaps_temporal", TEMP_DATA_DIR, TEMP_CODES, "temporal", ["--time-gaps"])

In [ ]:
run_tiger("tiger_time_embed_temporal", TEMP_DATA_DIR, TEMP_CODES, "temporal", ["--time-embed"])

## Summary

In [ ]:
rows = []
for path in sorted(glob.glob(f"{RUN_DIR}/{CATEGORY}_*.json")):
    d = json.load(open(path))
    cfg = d.get("config", {})
    test = d.get("test", {})
    diag = d.get("split_diagnostics", {})
    rows.append({
        "run": os.path.basename(path),
        "split": cfg.get("split"),
        "time_gaps": cfg.get("time_gaps"),
        "time_embed": cfg.get("time_embed"),
        "best_val_R@10": d.get("best_val_recall@10"),
        "test_R@5": test.get("recall@5"),
        "test_N@5": test.get("ndcg@5"),
        "test_R@10": test.get("recall@10"),
        "test_N@10": test.get("ndcg@10"),
        "invalid_code_rate": test.get("invalid_code_rate"),
        "decode_latency_s": test.get("decode_latency_s"),
        "test_seen_target_rate": (diag.get("test_target_coverage") or {}).get("seen_target_rate"),
        "path": path,
    })

summary_csv = f"{RUN_DIR}/summary_rq3_lr3e4_e100_seed{SEED}.csv"
summary_md = f"{RUN_DIR}/summary_rq3_lr3e4_e100_seed{SEED}.md"

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
    df.to_csv(summary_csv, index=False)
    md = df.to_markdown(index=False)
except Exception:
    md = "\n".join(str(r) for r in rows)

with open(summary_md, "w", encoding="utf-8") as f:
    f.write(md + "\n")

print("saved", summary_csv)
print("saved", summary_md)
print(md)